In [1]:
# Cell 1: Imports and load cached data
import polars as pl
import numpy as np
from pathlib import Path
# Run this in a notebook cell
%pip install scikit-learn

# Paths
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

# Load cached data
print("Loading cached data...")
all_events = pl.read_parquet(PROCESSED_DIR / "all_matches_with_zones.parquet")
matches = pl.read_parquet(PROCESSED_DIR / "matches_metadata.parquet")
player_metadata = pl.read_parquet(PROCESSED_DIR / "player_metadata.parquet")

print(f"Loaded {len(all_events):,} events")
print(f"Loaded {len(matches)} matches")
print(f"Loaded {len(player_metadata)} players")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 14.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.
Loading cached data...
Loaded 962,990 events
Loaded 306 matches
Loaded 507 players


### Event Attribute Exploration

Inspect available attributes and result values for each event type to inform weighting system design.

In [2]:
# Cell 2: Explore event attributes and available values
print("EXPLORING EVENT ATTRIBUTES")

# Check what attributes exist for different event types
event_types_to_check = ['PASS', 'SHOT', 'DUEL', 'CARRY', 'INTERCEPTION', 'CLEARANCE']

for event_type in event_types_to_check:
    print(f"\n{event_type}:")
    sample = all_events.filter(pl.col('event_type') == event_type).head(100)
    
    # Skip if no events of this type found
    if len(sample) == 0:
        print("  No events found")
        continue
    
    # Check what result values exist
    if 'result' in sample.columns:
        results = sample['result'].value_counts().sort('count', descending=True)
        result_list = results['result'].to_list()
        print(f"  Results: {result_list}")
    
    # Check pass types (only for PASS events)
    if event_type == 'PASS' and 'pass_type' in sample.columns:
        pass_types = sample['pass_type'].drop_nulls().value_counts().sort('count', descending=True)
        if len(pass_types) > 0:
            print(f"  Pass types: {pass_types['pass_type'].to_list()[:5]}")
    
    # Check duel types (only for DUEL events)
    if event_type == 'DUEL' and 'duel_type' in sample.columns:
        duel_types = sample['duel_type'].drop_nulls().value_counts().sort('count', descending=True)
        if len(duel_types) > 0:
            print(f"  Duel types: {duel_types['duel_type'].to_list()}")
    
    # Check success rate with null handling
    if 'success' in sample.columns:
        success_values = sample['success'].drop_nulls()
        if len(success_values) > 0:
            success_rate = success_values.mean()
            if success_rate is not None:  # FIX: Check for None before formatting
                print(f"  Success rate: {success_rate:.1%}")
            else:
                print(f"  Success rate: N/A (all null)")
        else:
            print(f"  Success rate: N/A (no success field)")

print("Event exploration complete")

EXPLORING EVENT ATTRIBUTES

PASS:
  Results: ['COMPLETE', 'INCOMPLETE']
  Pass types: ['HEAD_PASS', 'CHIPPED_PASS', 'HAND_PASS', 'SHOT_ASSIST']
  Success rate: 78.0%

SHOT:
  Results: ['OFF_TARGET', 'SAVED', 'GOAL', 'OWN_GOAL']
  Success rate: 12.0%

DUEL:
  Results: ['WON', 'LOST']
  Duel types: ['GROUND', 'AERIAL']
  Success rate: 50.0%

CARRY:
  Results: ['COMPLETE', 'INCOMPLETE']
  Success rate: 88.0%

INTERCEPTION:
  Results: ['SUCCESS']
  Success rate: 100.0%

CLEARANCE:
  Results: [None]
  Success rate: N/A (no success field)
Event exploration complete


### Define Transparent Weighting System

Assign explicit point values to all event types based on tactical impact, with higher weights for rare high-value actions (goals, assists) and zone-dependent scoring reflecting positional importance.

In [3]:
# Cell 3: Event weighting system
print("DEFINING EVENT WEIGHTING SYSTEM")

# Attack dimension: Actions that contribute to scoring
ATTACK_WEIGHTS = {
    'SHOT': {
        'GOAL': 10.0,                    # Scoring is most valuable
        'SAVED': 2.5,                    # On target, forced save
        'OFF_TARGET': 0.5,               # At least attempted
        # Note: 'BLOCKED' not in IMPECT data, removed
    },
    'PASS': {
        'SHOT_ASSIST': 8.0,              # Direct assist (pass leading to shot)
        'INTO_PENALTY_AREA': 2.0,        # Dangerous pass into box
        'PROGRESSIVE_COMPLETE': 1.5,     # Forward pass >10m
        'COMPLETE_ATTACKING': 1.0,       # Successful pass in attacking third
        'COMPLETE_MIDDLE': 0.5,          # Successful pass in middle third
        'COMPLETE_DEFENSIVE': 0.2,       # Successful pass in defensive third
        'INCOMPLETE': -0.3,              # Failed pass penalty
    },
    'CARRY': {
        'PROGRESSIVE_COMPLETE': 1.2,     # Dribble forward >10m
        'COMPLETE_ATTACKING': 0.8,       # Carry in attacking third
        'COMPLETE_MIDDLE': 0.4,          # Carry in middle third
        'INCOMPLETE': -0.2,              # Lost ball while carrying
    },
}

# Defense dimension: Actions that prevent goals
# Note: Duels are defensive actions, not included in attack weights to avoid double counting
DEFENSE_WEIGHTS = {
    'DUEL': {
        'WON_DEFENSIVE': 3.0,            # Critical - won ball in own third
        'WON_MIDDLE': 2.0,               # Important defensive action
        'WON_ATTACKING': 1.0,            # Pressing high up
        'LOST_DEFENSIVE': -1.0,          # Dangerous loss in own third
        'LOST_MIDDLE': -0.5,             # Loss in middle
        'LOST_ATTACKING': -0.2,          # Failed press
    },
    'INTERCEPTION': {
        'DEFENSIVE': 2.5,                # Critical interception
        'MIDDLE': 2.0,                   # Good defensive read
        'ATTACKING': 1.5,                # High press interception
    },
    'CLEARANCE': {
        'DEFENSIVE': 2.0,                # Clearing danger
        'MIDDLE': 1.0,                   # Clearing from midfield
    },
    'RECOVERY': {
        'DEFENSIVE': 1.5,                # Recovering loose ball in own third
        'MIDDLE': 1.0,                   # Recovery in middle
        'ATTACKING': 0.8,                # Recovery high up
    },
}

# Passing dimension: Passing quality and distribution
PASSING_WEIGHTS = {
    'PASS': {
        'COMPLETE_LONG': 1.5,            # Long pass >30m completed
        'COMPLETE_PROGRESSIVE': 1.2,     # Progressive pass >10m
        'COMPLETE': 0.5,                 # Any successful pass
        'INCOMPLETE_LONG': -0.5,         # Failed long pass
        'INCOMPLETE': -0.3,              # Failed pass
        'SHOT_ASSIST': 3.0,              # Key pass (in passing context too)
    },
}

# Position-specific weighting: How dimensions combine based on tactical role
POSITION_WEIGHTS = {
    'forward': {'attack': 0.60, 'passing': 0.30, 'defense': 0.10},
    'midfielder': {'attack': 0.35, 'passing': 0.35, 'defense': 0.30},
    'defender': {'attack': 0.15, 'passing': 0.25, 'defense': 0.60},
    'goalkeeper': {'attack': 0.05, 'passing': 0.25, 'defense': 0.70},
}

print("\nWeighting system components:")
print(f"  Attack events: {len(ATTACK_WEIGHTS)} categories")
print(f"  Defense events: {len(DEFENSE_WEIGHTS)} categories")
print(f"  Passing events: {len(PASSING_WEIGHTS)} categories")
print(f"  Position profiles: {len(POSITION_WEIGHTS)} roles")

print("Weighting system defined")

DEFINING EVENT WEIGHTING SYSTEM

Weighting system components:
  Attack events: 3 categories
  Defense events: 4 categories
  Passing events: 1 categories
  Position profiles: 4 roles
Weighting system defined


### Helper Functions for Event Classification

Define utility functions to identify special event characteristics: progressive actions (>10m forward), long passes (>30m), and passes into the penalty area for enhanced scoring granularity.

In [4]:
# Cell 4: Helper functions for event classification
def is_progressive_pass(row):
    """
    Check if a pass is progressive (forward >10m toward opponent's goal).
    Accounts for team attack direction.
    """
    if row['end_coordinates_x'] is None or row['coordinates_x'] is None:
        return False
    if 'home_squad_id' not in row or row['home_squad_id'] is None:
        return False
    
    # Calculate distance
    distance = row['end_coordinates_x'] - row['coordinates_x']
    
    # Determine if home team
    is_home = (str(row['team_id']) == str(row['home_squad_id']))
    
    # Home team: positive distance = forward
    # Away team: negative distance = forward
    if is_home:
        return distance >= 10.0
    else:
        return distance <= -10.0

def is_progressive_carry(row):
    """
    Check if a carry is progressive (forward >10m toward opponent's goal).
    Accounts for team attack direction.
    """
    if row['end_coordinates_x'] is None or row['coordinates_x'] is None:
        return False
    if 'home_squad_id' not in row or row['home_squad_id'] is None:
        return False
    
    # Calculate distance
    distance = row['end_coordinates_x'] - row['coordinates_x']
    
    # Determine if home team
    is_home = (str(row['team_id']) == str(row['home_squad_id']))
    
    # Home team: positive distance = forward
    # Away team: negative distance = forward
    if is_home:
        return distance >= 10.0
    else:
        return distance <= -10.0

def is_long_pass(row):
    """
    Check if a pass is long (total distance >30m).
    Direction-agnostic - just measures total distance.
    """
    if row['end_coordinates_x'] is None or row['coordinates_x'] is None:
        return False
    if row['end_coordinates_y'] is None or row['coordinates_y'] is None:
        return False
    
    distance = np.sqrt(
        (row['end_coordinates_x'] - row['coordinates_x'])**2 + 
        (row['end_coordinates_y'] - row['coordinates_y'])**2
    )
    return distance >= 30.0

def is_into_penalty_area(row):
    """
    Check if pass ends in penalty area.
    
    For home team: penalty area at x > 35.5 (attacking right)
    For away team: penalty area at x < -35.5 (attacking left)
    Both: within -9.15 < y < 9.15 (box width)
    """
    if row['end_coordinates_x'] is None or row['end_coordinates_y'] is None:
        return False
    if 'home_squad_id' not in row or row['home_squad_id'] is None:
        return False
    
    # Check if home team
    is_home = (str(row['team_id']) == str(row['home_squad_id']))
    
    # Check y-coordinate (same for both teams)
    in_penalty_y = -9.15 < row['end_coordinates_y'] < 9.15
    
    # Check x-coordinate (different for home vs away)
    if is_home:
        in_penalty_x = row['end_coordinates_x'] > 35.5  # Right side box
    else:
        in_penalty_x = row['end_coordinates_x'] < -35.5  # Left side box
    
    return in_penalty_x and in_penalty_y

print("Helper functions defined:")
print("  - is_progressive_pass (>10m forward, team-aware)")
print("  - is_progressive_carry (>10m forward, team-aware)")
print("  - is_long_pass (>30m total distance)")
print("  - is_into_penalty_area (ends in box, team-aware)")

Helper functions defined:
  - is_progressive_pass (>10m forward, team-aware)
  - is_progressive_carry (>10m forward, team-aware)
  - is_long_pass (>30m total distance)
  - is_into_penalty_area (ends in box, team-aware)


### Calculate Attack Points

Implement attack scoring function that assigns points to shots, passes, and carries based on outcome and zone, with bonus points for high-value actions like goals, assists, and progressive plays.


In [5]:
# Cell 5: Calculate attack points for each event
def calculate_attack_points(row):
    """
    Calculate attack rating points for a single event.
    Returns points based on event type, result, and zone.
    """
    event_type = row['event_type']
    result = row['result']
    zone = row['zone']
    
    points = 0.0
    
    # SHOTS - Most direct attacking contribution
    if event_type == 'SHOT':
        if result == 'GOAL':
            points = ATTACK_WEIGHTS['SHOT']['GOAL']
        elif result == 'SAVED':
            points = ATTACK_WEIGHTS['SHOT']['SAVED']
        elif result == 'OFF_TARGET':
            points = ATTACK_WEIGHTS['SHOT']['OFF_TARGET']
        # Note: 'BLOCKED' removed (not in data)
    
    # PASSES - Creating chances and progressing play
    elif event_type == 'PASS':
        pass_type = row.get('pass_type', None)
        
        # Check for assist (highest priority)
        if pass_type == 'SHOT_ASSIST':
            points = ATTACK_WEIGHTS['PASS']['SHOT_ASSIST']
        
        # Check if into penalty area
        elif is_into_penalty_area(row):
            points = ATTACK_WEIGHTS['PASS']['INTO_PENALTY_AREA']
        
        # Check if progressive
        elif result == 'COMPLETE' and is_progressive_pass(row):
            points = ATTACK_WEIGHTS['PASS']['PROGRESSIVE_COMPLETE']
        
        # Zone-based scoring for complete passes
        elif result == 'COMPLETE':
            if zone == 'attacking_third':
                points = ATTACK_WEIGHTS['PASS']['COMPLETE_ATTACKING']
            elif zone == 'middle_third':
                points = ATTACK_WEIGHTS['PASS']['COMPLETE_MIDDLE']
            elif zone == 'defensive_third':
                points = ATTACK_WEIGHTS['PASS']['COMPLETE_DEFENSIVE']
        
        # Failed pass penalty
        elif result == 'INCOMPLETE':
            points = ATTACK_WEIGHTS['PASS']['INCOMPLETE']
    
    # CARRIES - Dribbling and ball progression
    elif event_type == 'CARRY':
        if result == 'COMPLETE':
            if is_progressive_carry(row):
                points = ATTACK_WEIGHTS['CARRY']['PROGRESSIVE_COMPLETE']
            elif zone == 'attacking_third':
                points = ATTACK_WEIGHTS['CARRY']['COMPLETE_ATTACKING']
            elif zone == 'middle_third':
                points = ATTACK_WEIGHTS['CARRY']['COMPLETE_MIDDLE']
        elif result == 'INCOMPLETE':
            points = ATTACK_WEIGHTS['CARRY']['INCOMPLETE']
    
    # Note: DUELS removed from attack scoring to prevent double-counting
    # Duels are scored only in defense dimension
    
    return points

# Test the function
print("Testing attack points calculation on sample events...")

test_events = all_events.filter(
    pl.col('event_type').is_in(['PASS', 'SHOT', 'CARRY'])
).sample(5)

for row in test_events.iter_rows(named=True):
    pts = calculate_attack_points(row)
    print(f"{row['event_type']:15s} | {str(row['result']):15s} | {row['zone']:20s} | Points: {pts:>5.2f}")

print("\nAttack points function ready")

Testing attack points calculation on sample events...
CARRY           | COMPLETE        | middle_third         | Points:  0.40
CARRY           | COMPLETE        | middle_third         | Points:  0.40
CARRY           | COMPLETE        | middle_third         | Points:  0.40
CARRY           | COMPLETE        | attacking_third      | Points:  0.80
PASS            | COMPLETE        | middle_third         | Points:  0.50

Attack points function ready


### Calculate Defense Points

Implement defense scoring function that assigns points to duels, interceptions, clearances, and recoveries, with higher weights for actions in defensive third where stakes are highest.

In [6]:
# Cell 6: Calculate defense points
def calculate_defense_points(row):
    """
    Calculate defense rating points for a single event.
    """
    event_type = row['event_type']
    result = row['result']
    zone = row['zone']
    
    points = 0.0
    
    # DUELS (defensive contribution)
    if event_type == 'DUEL':
        if result == 'WON':
            if zone == 'defensive_third':
                points = DEFENSE_WEIGHTS['DUEL']['WON_DEFENSIVE']
            elif zone == 'middle_third':
                points = DEFENSE_WEIGHTS['DUEL']['WON_MIDDLE']
            elif zone == 'attacking_third':
                points = DEFENSE_WEIGHTS['DUEL']['WON_ATTACKING']
        elif result == 'LOST':
            if zone == 'defensive_third':
                points = DEFENSE_WEIGHTS['DUEL']['LOST_DEFENSIVE']
            elif zone == 'middle_third':
                points = DEFENSE_WEIGHTS['DUEL']['LOST_MIDDLE']
            elif zone == 'attacking_third':
                points = DEFENSE_WEIGHTS['DUEL']['LOST_ATTACKING']
    
    # INTERCEPTIONS
    elif event_type == 'INTERCEPTION':
        if zone == 'defensive_third':
            points = DEFENSE_WEIGHTS['INTERCEPTION']['DEFENSIVE']
        elif zone == 'middle_third':
            points = DEFENSE_WEIGHTS['INTERCEPTION']['MIDDLE']
        elif zone == 'attacking_third':
            points = DEFENSE_WEIGHTS['INTERCEPTION']['ATTACKING']
    
    # CLEARANCES
    elif event_type == 'CLEARANCE':
        if zone == 'defensive_third':
            points = DEFENSE_WEIGHTS['CLEARANCE']['DEFENSIVE']
        elif zone == 'middle_third':
            points = DEFENSE_WEIGHTS['CLEARANCE']['MIDDLE']
    
    # RECOVERIES
    elif event_type == 'RECOVERY':
        if zone == 'defensive_third':
            points = DEFENSE_WEIGHTS['RECOVERY']['DEFENSIVE']
        elif zone == 'middle_third':
            points = DEFENSE_WEIGHTS['RECOVERY']['MIDDLE']
        elif zone == 'attacking_third':
            points = DEFENSE_WEIGHTS['RECOVERY']['ATTACKING']
    
    return points

print("Testing defense points calculation on sample events...")

# Test on defensive events
test_events = all_events.filter(
    pl.col('event_type').is_in(['DUEL', 'INTERCEPTION', 'CLEARANCE', 'RECOVERY'])
).sample(5)

for row in test_events.iter_rows(named=True):
    pts = calculate_defense_points(row)
    print(f"{row['event_type']:15s} | {str(row['result']):15s} | {row['zone']:20s} | Points: {pts:>5.2f}")

print("\n Defense points function ready")

Testing defense points calculation on sample events...
RECOVERY        | None            | middle_third         | Points:  1.00
RECOVERY        | None            | defensive_third      | Points:  1.50
RECOVERY        | None            | defensive_third      | Points:  1.50
DUEL            | LOST            | attacking_third      | Points: -0.20
RECOVERY        | None            | middle_third         | Points:  1.00

 Defense points function ready


### Calculate Passing Points

Implement passing quality scoring that rewards successful distribution (complete passes, long passes, progressive passes) and penalizes turnovers, with special bonuses for assists.

In [7]:
# Cell 7: Calculate passing points for each event
def calculate_passing_points(row):
    """
    Calculate passing rating points for a single event.
    Focuses on pass completion, distance, and progression quality.
    """
    event_type = row['event_type']
    result = row['result']
    
    points = 0.0
    
    # Only PASS events contribute to passing rating
    if event_type == 'PASS':
        pass_type = row.get('pass_type', None)
        
        # Assist bonus (highest priority)
        if pass_type == 'SHOT_ASSIST':
            points = PASSING_WEIGHTS['PASS']['SHOT_ASSIST']
        
        # Complete passes - evaluate quality
        elif result == 'COMPLETE':
            # Long pass bonus
            if is_long_pass(row):
                points = PASSING_WEIGHTS['PASS']['COMPLETE_LONG']
            # Progressive pass bonus
            elif is_progressive_pass(row):
                points = PASSING_WEIGHTS['PASS']['COMPLETE_PROGRESSIVE']
            # Regular complete pass
            else:
                points = PASSING_WEIGHTS['PASS']['COMPLETE']
        
        # Failed passes - penalize turnovers
        elif result == 'INCOMPLETE':
            # Note: We don't check is_long_pass for incomplete because
            # end_coordinates may be unreliable (interception point, not target)
            # Apply standard incomplete penalty
            points = PASSING_WEIGHTS['PASS']['INCOMPLETE']
    
    return points

# Test the function
print("Testing passing points calculation on sample events...")

test_events = all_events.filter(
    pl.col('event_type') == 'PASS'
).sample(5)

for row in test_events.iter_rows(named=True):
    pts = calculate_passing_points(row)
    is_prog = is_progressive_pass(row)
    is_long = is_long_pass(row) if row['result'] == 'COMPLETE' else False
    print(f"{row['result']:15s} | Progressive: {str(is_prog):5s} | Long: {str(is_long):5s} | Points: {pts:>5.2f}")

print("\nPassing points function ready")

Testing passing points calculation on sample events...
COMPLETE        | Progressive: False | Long: False | Points:  0.50
INCOMPLETE      | Progressive: False | Long: False | Points: -0.30
INCOMPLETE      | Progressive: False | Long: False | Points: -0.30
COMPLETE        | Progressive: False | Long: False | Points:  0.50
COMPLETE        | Progressive: False | Long: False | Points:  0.50

Passing points function ready


### Apply Scoring to All Events

Calculate attack, defense, and passing points for all 962,990 events by applying the scoring functions, then add these values as new columns for player-level aggregation.


In [8]:
# Cell 8: Apply scoring functions to all events
print("Calculating points for all 962,990 events...")

import time
start_time = time.time()

# Convert to list of dicts for iteration
events_list = all_events.to_dicts()

# Initialize point lists
attack_points_list = []
defense_points_list = []
passing_points_list = []

# Calculate points for each event
for i, row in enumerate(events_list):
    attack_points_list.append(calculate_attack_points(row))
    defense_points_list.append(calculate_defense_points(row))
    passing_points_list.append(calculate_passing_points(row))
    
    # Progress indicator every 100k events
    if (i + 1) % 100000 == 0:
        elapsed = time.time() - start_time
        progress = (i + 1) / len(events_list)
        remaining = (elapsed / progress) * (1 - progress)
        print(f"  Processed {i+1:,}/{len(events_list):,} events ({progress:.1%}) | "
              f"Elapsed: {elapsed:.1f}s | Remaining: {remaining:.1f}s")

# Add points as new columns
all_events = all_events.with_columns([
    pl.Series("attack_points", attack_points_list),
    pl.Series("defense_points", defense_points_list),
    pl.Series("passing_points", passing_points_list),
])

elapsed_total = time.time() - start_time
print(f"\nCalculated points for all events in {elapsed_total:.1f} seconds")

# Display summary statistics
print("\nPOINTS DISTRIBUTION SUMMARY")

print("\nAttack Points:")
print(f"  Total: {all_events['attack_points'].sum():,.0f}")
print(f"  Mean: {all_events['attack_points'].mean():.3f}")
print(f"  Positive events: {(all_events['attack_points'] > 0).sum():,}")
print(f"  Negative events: {(all_events['attack_points'] < 0).sum():,}")

print("\nDefense Points:")
print(f"  Total: {all_events['defense_points'].sum():,.0f}")
print(f"  Mean: {all_events['defense_points'].mean():.3f}")
print(f"  Positive events: {(all_events['defense_points'] > 0).sum():,}")
print(f"  Negative events: {(all_events['defense_points'] < 0).sum():,}")

print("\nPassing Points:")
print(f"  Total: {all_events['passing_points'].sum():,.0f}")
print(f"  Mean: {all_events['passing_points'].mean():.3f}")
print(f"  Positive events: {(all_events['passing_points'] > 0).sum():,}")
print(f"  Negative events: {(all_events['passing_points'] < 0).sum():,}")

# Display sample events with calculated points
print("SAMPLE EVENTS WITH POINTS")
print(all_events.select([
    'event_type', 'result', 'zone', 
    'attack_points', 'defense_points', 'passing_points'
]).sample(10))

print("Event scoring complete")

Calculating points for all 962,990 events...
  Processed 100,000/962,990 events (10.4%) | Elapsed: 4.0s | Remaining: 34.7s
  Processed 200,000/962,990 events (20.8%) | Elapsed: 4.2s | Remaining: 15.8s
  Processed 300,000/962,990 events (31.2%) | Elapsed: 4.3s | Remaining: 9.5s
  Processed 400,000/962,990 events (41.5%) | Elapsed: 4.4s | Remaining: 6.3s
  Processed 500,000/962,990 events (51.9%) | Elapsed: 4.6s | Remaining: 4.3s
  Processed 600,000/962,990 events (62.3%) | Elapsed: 4.8s | Remaining: 2.9s
  Processed 700,000/962,990 events (72.7%) | Elapsed: 4.9s | Remaining: 1.9s
  Processed 800,000/962,990 events (83.1%) | Elapsed: 5.1s | Remaining: 1.0s
  Processed 900,000/962,990 events (93.5%) | Elapsed: 5.3s | Remaining: 0.4s

Calculated points for all events in 5.5 seconds

POINTS DISTRIBUTION SUMMARY

Attack Points:
  Total: 294,752
  Mean: 0.306
  Positive events: 405,459
  Negative events: 92,241

Defense Points:
  Total: 122,813
  Mean: 0.128
  Positive events: 91,235
  Negati

In [9]:
# Cell 9: Save processed events
print("Saving events with calculated points...")

all_events.write_parquet(PROCESSED_DIR / "all_events_with_points.parquet")

print(f"Saved to: {PROCESSED_DIR / 'all_events_with_points.parquet'}")
print(f"   File size: {(PROCESSED_DIR / 'all_events_with_points.parquet').stat().st_size / 1_000_000:.1f} MB")

Saving events with calculated points...
Saved to: ../data/processed/all_events_with_points.parquet
   File size: 16.9 MB


### Aggregate Points by Player

Group all events by player and sum points across attack, defense, and passing dimensions to create raw player ratings before normalization.

In [10]:
# Cell 10: Aggregate points by player
print("Aggregating points by player...")

player_ratings = (
    all_events
    .filter(pl.col('player_id').is_not_null())  # Remove null player_ids
    .group_by(['player_id', 'team_id'])
    .agg([
        # Sum all points
        pl.sum('attack_points').alias('total_attack_points'),
        pl.sum('defense_points').alias('total_defense_points'),
        pl.sum('passing_points').alias('total_passing_points'),
        
        # Count events for diagnostics
        pl.len().alias('total_events'),
        pl.col('match_id').n_unique().alias('matches_played'),
        
        # Calculate average field position for position classification
        pl.mean('coordinates_x').alias('avg_x_position'),
        pl.mean('coordinates_y').alias('avg_y_position'),
    ])
    .sort('total_attack_points', descending=True)
)

print(f"\nAggregated ratings for {len(player_ratings)} players")
print(f"\nTop 10 players by attack points:")
print(player_ratings.select(['player_id', 'team_id', 'total_attack_points', 'matches_played']).head(10))

print(f"\nTop 10 players by defense points:")
print(player_ratings.sort('total_defense_points', descending=True).select(['player_id', 'team_id', 'total_defense_points', 'matches_played']).head(10))

print(f"\nTop 10 players by passing points:")
print(player_ratings.sort('total_passing_points', descending=True).select(['player_id', 'team_id', 'total_passing_points', 'matches_played']).head(10))

Aggregating points by player...

Aggregated ratings for 506 players

Top 10 players by attack points:
shape: (10, 4)
┌───────────┬─────────┬─────────────────────┬────────────────┐
│ player_id ┆ team_id ┆ total_attack_points ┆ matches_played │
│ ---       ┆ ---     ┆ ---                 ┆ ---            │
│ str       ┆ str     ┆ f64                 ┆ u32            │
╞═══════════╪═════════╪═════════════════════╪════════════════╡
│ 281       ┆ 41      ┆ 3090.6              ┆ 33             │
│ 32214     ┆ 29      ┆ 2675.4              ┆ 33             │
│ 98        ┆ 33      ┆ 2486.7              ┆ 28             │
│ 1359      ┆ 46      ┆ 2440.3              ┆ 33             │
│ 27123     ┆ 46      ┆ 2418.3              ┆ 31             │
│ 5616      ┆ 41      ┆ 2403.2              ┆ 33             │
│ 13823     ┆ 37      ┆ 2173.9              ┆ 31             │
│ 57297     ┆ 35      ┆ 2140.6              ┆ 33             │
│ 50321     ┆ 33      ┆ 2090.0              ┆ 25             │
│

## Extract Player Names

Load player metadata from sample matches to map player IDs to actual names, enabling readable leaderboards and analysis.


In [11]:
# Cell 10A: Extract player names from match metadata
from kloppy import impect
from tqdm.notebook import tqdm

print("Extracting player names from match metadata...")

# Load one match to understand metadata structure
sample_match_id = matches['matchId'][0]
sample_dataset = impect.load_open_data(match_id=sample_match_id, competition_id=743)

print("\nDataset metadata structure:")
print(f"  Teams per match: {len(sample_dataset.metadata.teams)}")
print(f"  Periods per match: {len(sample_dataset.metadata.periods)}")

# Extract player information from sample of matches
player_names_dict = {}

print("\nSampling 10 matches to extract player names...")
for match_id in tqdm(matches['matchId'][:10], desc="Loading matches"):
    try:
        dataset = impect.load_open_data(match_id=match_id, competition_id=743)
        
        # Get players from both teams
        for team in dataset.metadata.teams:
            for player in team.players:
                if player.player_id not in player_names_dict:
                    player_names_dict[player.player_id] = {
                        'player_name': player.name if hasattr(player, 'name') else f"Player_{player.player_id}",
                        'jersey_no': player.jersey_no if hasattr(player, 'jersey_no') else None
                    }
    except Exception as e:
        continue

print(f"\nExtracted names for {len(player_names_dict)} players")

# Convert to dataframe
player_names_df = pl.DataFrame([
    {'player_id': str(pid), 'player_name': info['player_name'], 'jersey_no': info['jersey_no']}
    for pid, info in player_names_dict.items()
])

# Join with player ratings
player_ratings = player_ratings.join(
    player_names_df,
    on='player_id',
    how='left'
)

print("TOP 10 PLAYERS BY DIMENSION (With Names)")


print("\nTop 10 by attack points:")
print(player_ratings.select([
    'player_name', 'player_id', 'total_attack_points', 'matches_played'
]).head(10))

print("\nTop 10 by defense points:")
print(player_ratings.sort('total_defense_points', descending=True).select([
    'player_name', 'player_id', 'total_defense_points', 'matches_played'
]).head(10))

print("\nTop 10 by passing points:")
print(player_ratings.sort('total_passing_points', descending=True).select([
    'player_name', 'player_id', 'total_passing_points', 'matches_played'
]).head(10))

print("Player name extraction complete")


Extracting player names from match metadata...


/Users/tanishbhilare/anaconda3/envs/soccer-hackathon/lib/python3.11/site-packages/kloppy/_providers/impect.py:88: UserWarning: 

You are about to use IMPECT public data.
By using this data, you are agreeing to the user agreement. 
The user agreement can be found here: https://github.com/ImpectAPI/open-data/blob/main/LICENSE.pdf

  warnings.warn(



Dataset metadata structure:
  Teams per match: 2
  Periods per match: 2

Sampling 10 matches to extract player names...


Loading matches:   0%|          | 0/10 [00:00<?, ?it/s]


Extracted names for 363 players
TOP 10 PLAYERS BY DIMENSION (With Names)

Top 10 by attack points:
shape: (10, 4)
┌────────────────────┬───────────┬─────────────────────┬────────────────┐
│ player_name        ┆ player_id ┆ total_attack_points ┆ matches_played │
│ ---                ┆ ---       ┆ ---                 ┆ ---            │
│ str                ┆ str       ┆ f64                 ┆ u32            │
╞════════════════════╪═══════════╪═════════════════════╪════════════════╡
│ Granit Xhaka       ┆ 281       ┆ 3090.6              ┆ 33             │
│ Nico Schlotterbeck ┆ 32214     ┆ 2675.4              ┆ 33             │
│ Joshua Kimmich     ┆ 98        ┆ 2486.7              ┆ 28             │
│ Waldemar Anton     ┆ 1359      ┆ 2440.3              ┆ 33             │
│ Angelo Stiller     ┆ 27123     ┆ 2418.3              ┆ 31             │
│ Alejandro Grimaldo ┆ 5616      ┆ 2403.2              ┆ 33             │
│ David Raum         ┆ 13823     ┆ 2173.9              ┆ 31            

### Investigate Null Player Names

Identify and analyze players without extracted names to understand data coverage from the sample of matches used for name extraction.

In [12]:
# Cell: Investigate null-named players
print("Investigating null-named players...")
print("="*60)

# Find players with null names
null_in_ratings = player_ratings.filter(pl.col('player_name').is_null())
print(f"\nPlayers with null names: {len(null_in_ratings)}")

if len(null_in_ratings) > 0:
    print("\nTop 5 null-named players by total events:")
    print(null_in_ratings.select([
        'player_id', 'team_id', 'matches_played', 'total_events'
    ]).sort('total_events', descending=True).head(5))
    
    # Check one example
    sample_null_id = null_in_ratings['player_id'][0]
    null_events = all_events.filter(pl.col('player_id') == sample_null_id)
    
    print(f"\nExample null player (ID={sample_null_id}):")
    print(f"  Total events: {len(null_events)}")
    print(f"  Matches played: {null_events['match_id'].n_unique()}")
    print(f"  Average position: ({null_events['coordinates_x'].mean():.2f}, {null_events['coordinates_y'].mean():.2f})")
else:
    print("All players have names!")


Investigating null-named players...

Players with null names: 151

Top 5 null-named players by total events:
shape: (5, 4)
┌───────────┬─────────┬────────────────┬──────────────┐
│ player_id ┆ team_id ┆ matches_played ┆ total_events │
│ ---       ┆ ---     ┆ ---            ┆ ---          │
│ str       ┆ str     ┆ u32            ┆ u32          │
╞═══════════╪═════════╪════════════════╪══════════════╡
│ 568       ┆ 32      ┆ 30             ┆ 5113         │
│ 1047      ┆ 39      ┆ 30             ┆ 4509         │
│ 1333      ┆ 30      ┆ 30             ┆ 4387         │
│ 53110     ┆ 41      ┆ 25             ┆ 4230         │
│ 1050      ┆ 34      ┆ 28             ┆ 3434         │
└───────────┴─────────┴────────────────┴──────────────┘



Example null player (ID=568):
  Total events: 5113
  Matches played: 30
  Average position: (-16.88, -3.35)


In [13]:
print("\nTop 10 by attack points (with names):")
print(player_ratings.select(['player_name', 'player_id', 'total_attack_points', 'matches_played']).head(10))
print(f"\nTop 10 players by defense points:")
print(player_ratings.sort('total_defense_points', descending=True).select(['player_name','player_id', 'team_id', 'total_defense_points', 'matches_played']).head(10))

print(f"\nTop 10 players by passing points:")
print(player_ratings.sort('total_passing_points', descending=True).select(['player_name','player_id', 'team_id', 'total_passing_points', 'matches_played']).head(10))


Top 10 by attack points (with names):
shape: (10, 4)
┌────────────────────┬───────────┬─────────────────────┬────────────────┐
│ player_name        ┆ player_id ┆ total_attack_points ┆ matches_played │
│ ---                ┆ ---       ┆ ---                 ┆ ---            │
│ str                ┆ str       ┆ f64                 ┆ u32            │
╞════════════════════╪═══════════╪═════════════════════╪════════════════╡
│ Granit Xhaka       ┆ 281       ┆ 3090.6              ┆ 33             │
│ Nico Schlotterbeck ┆ 32214     ┆ 2675.4              ┆ 33             │
│ Joshua Kimmich     ┆ 98        ┆ 2486.7              ┆ 28             │
│ Waldemar Anton     ┆ 1359      ┆ 2440.3              ┆ 33             │
│ Angelo Stiller     ┆ 27123     ┆ 2418.3              ┆ 31             │
│ Alejandro Grimaldo ┆ 5616      ┆ 2403.2              ┆ 33             │
│ David Raum         ┆ 13823     ┆ 2173.9              ┆ 31             │
│ Willian Pacho      ┆ 57297     ┆ 2140.6              ┆ 3

### Position Classification

Classify players into forwards, midfielders, and defenders based on average field position, enabling position-specific weighting in overall rating calculation.

In [14]:
# Cell 11: Classify player positions based on average field position
print("Classifying player positions based on average field position...")

def classify_position(avg_x, attack_ratio, defense_ratio):
    """
    Classify player position based on average x-coordinate.
    
    Position zones (secondspectrum coordinates):
    - Forward: avg_x > 10 (spends time in attacking areas)
    - Midfielder: -10 <= avg_x <= 10 (spends time in middle)
    - Defender: avg_x < -10 (spends time in defensive areas)
    
    Note: attack_ratio and defense_ratio available for future refinement
    """
    
    # Primary classification based on field position
    if avg_x > 10:
        return 'forward'
    elif avg_x < -10:
        return 'defender'
    else:
        return 'midfielder'

# Calculate total points for ratio calculations
player_ratings = player_ratings.with_columns([
    (pl.col('total_attack_points') + 
     pl.col('total_defense_points') + 
     pl.col('total_passing_points')).alias('total_points'),
])

# Calculate attack and defense ratios (proportion of total points)
player_ratings = player_ratings.with_columns([
    (pl.col('total_attack_points') / pl.col('total_points')).fill_nan(0).alias('attack_ratio'),
    (pl.col('total_defense_points') / pl.col('total_points')).fill_nan(0).alias('defense_ratio'),
])

# Apply position classification to all players
positions = []
for row in player_ratings.iter_rows(named=True):
    pos = classify_position(
        row['avg_x_position'] if row['avg_x_position'] is not None else 0,
        row['attack_ratio'],
        row['defense_ratio']
    )
    positions.append(pos)

player_ratings = player_ratings.with_columns([
    pl.Series('position', positions)
])

# Display position distribution
print("\nPosition distribution:")
position_summary = player_ratings.group_by('position').agg([
    pl.len().alias('count'),
    pl.mean('avg_x_position').alias('avg_x'),
    pl.mean('attack_ratio').alias('avg_attack_ratio'),
    pl.mean('defense_ratio').alias('avg_defense_ratio'),
]).sort('position')

print(position_summary)


print("Position classification complete")


# Show examples from each position
print("\nSample players by position:")
for pos in ['defender', 'midfielder', 'forward']:
    print(f"\n{pos.upper()}S (sample of 3):")
    sample = player_ratings.filter(pl.col('position') == pos).select([
        'player_name', 'avg_x_position', 'attack_ratio', 'defense_ratio', 'matches_played'
    ]).head(3)
    print(sample)

Classifying player positions based on average field position...

Position distribution:
shape: (3, 5)
┌────────────┬───────┬────────────┬──────────────────┬───────────────────┐
│ position   ┆ count ┆ avg_x      ┆ avg_attack_ratio ┆ avg_defense_ratio │
│ ---        ┆ ---   ┆ ---        ┆ ---              ┆ ---               │
│ str        ┆ u32   ┆ f64        ┆ f64              ┆ f64               │
╞════════════╪═══════╪════════════╪══════════════════╪═══════════════════╡
│ defender   ┆ 123   ┆ -22.713444 ┆ 0.492531         ┆ 0.222627          │
│ forward    ┆ 127   ┆ 14.075714  ┆ 0.613828         ┆ 0.186693          │
│ midfielder ┆ 256   ┆ 1.165859   ┆ 0.51169          ┆ 0.252541          │
└────────────┴───────┴────────────┴──────────────────┴───────────────────┘
Position classification complete

Sample players by position:

DEFENDERS (sample of 3):
shape: (3, 5)
┌────────────────────┬────────────────┬──────────────┬───────────────┬────────────────┐
│ player_name        ┆ avg_x_posi

## Estimate Playing Time and Filter Sample

Estimate minutes played using event participation rate as a proxy, then filter to players with at least 500 minutes to ensure statistical significance in ratings.

In [15]:
# Cell 12: Estimate playing time and filter for minimum minutes
print("Estimating minutes played per player...")

# Calculate average events per match across entire dataset
match_stats = all_events.group_by('match_id').agg([
    pl.len().alias('total_events'),
])

avg_events_per_match = match_stats['total_events'].mean()
print(f"Average events per match: {avg_events_per_match:.0f}")

# Calculate events per match for each player
player_ratings = player_ratings.with_columns([
    (pl.col('total_events') / pl.col('matches_played')).alias('events_per_match')
])

# Estimate minutes per match
# Heuristic: A starter with ~35 events plays ~90 minutes
# Scale linearly and cap at 90
avg_starter_events = 35

player_ratings = player_ratings.with_columns([
    (
        (pl.col('events_per_match') / avg_starter_events) * 90
    ).clip(0, 90).alias('estimated_minutes_per_match')
])

# Calculate total minutes across season
player_ratings = player_ratings.with_columns([
    (pl.col('estimated_minutes_per_match') * pl.col('matches_played')).alias('estimated_total_minutes')
])

# Display distribution statistics
print("\nEstimated minutes distribution:")
print(f"  Minimum: {player_ratings['estimated_total_minutes'].min():.0f} min")
print(f"  Maximum: {player_ratings['estimated_total_minutes'].max():.0f} min")
print(f"  Mean: {player_ratings['estimated_total_minutes'].mean():.0f} min")
print(f"  Median: {player_ratings['estimated_total_minutes'].median():.0f} min")

# Filter for players with significant playing time
player_ratings_filtered = player_ratings.filter(pl.col('estimated_total_minutes') >= 500)

print(f"\nPlayers by playing time threshold:")
print(f"  Total players: {len(player_ratings)}")
print(f"  >500 minutes: {len(player_ratings_filtered)} ({len(player_ratings_filtered)/len(player_ratings):.1%})")
print(f"  >1000 minutes: {(player_ratings['estimated_total_minutes'] >= 1000).sum()} ({(player_ratings['estimated_total_minutes'] >= 1000).sum()/len(player_ratings):.1%})")
print(f"  >2000 minutes: {(player_ratings['estimated_total_minutes'] >= 2000).sum()} ({(player_ratings['estimated_total_minutes'] >= 2000).sum()/len(player_ratings):.1%})")

# Show top players by estimated minutes
print("TOP 10 PLAYERS BY ESTIMATED PLAYING TIME")
print(player_ratings.select([
    'player_name', 'player_id', 'matches_played', 'total_events', 
    'events_per_match', 'estimated_minutes_per_match', 'estimated_total_minutes'
]).sort('estimated_total_minutes', descending=True).head(10))


print(f"Filtered to {len(player_ratings_filtered)} players with >500 minutes")

Estimating minutes played per player...
Average events per match: 3147

Estimated minutes distribution:
  Minimum: 8 min
  Maximum: 3060 min
  Mean: 1650 min
  Median: 1890 min

Players by playing time threshold:
  Total players: 506
  >500 minutes: 396 (78.3%)
  >1000 minutes: 347 (68.6%)
  >2000 minutes: 229 (45.3%)
TOP 10 PLAYERS BY ESTIMATED PLAYING TIME
shape: (10, 7)
┌──────────────┬───────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┐
│ player_name  ┆ player_id ┆ matches_play ┆ total_events ┆ events_per_ ┆ estimated_m ┆ estimated_t │
│ ---          ┆ ---       ┆ ed           ┆ ---          ┆ match       ┆ inutes_per_ ┆ otal_minute │
│ str          ┆ str       ┆ ---          ┆ u32          ┆ ---         ┆ match       ┆ s           │
│              ┆           ┆ u32          ┆              ┆ f64         ┆ ---         ┆ ---         │
│              ┆           ┆              ┆              ┆             ┆ f64         ┆ f64         │
╞══════════════╪══

### Normalize to Per-90 Minutes

Convert raw point totals to per-90-minute rates to enable fair comparison between players with different playing time, filtering to players with at least 500 minutes for statistical reliability.

In [16]:
# Cell 13: Normalize to per-90 minutes
print("Normalizing ratings to per-90 minutes...")

# Calculate per-90 ratings
player_ratings = player_ratings.with_columns([
    ((pl.col('total_attack_points') / pl.col('estimated_total_minutes')) * 90).alias('attack_per90'),
    ((pl.col('total_defense_points') / pl.col('estimated_total_minutes')) * 90).alias('defense_per90'),
    ((pl.col('total_passing_points') / pl.col('estimated_total_minutes')) * 90).alias('passing_per90'),
])

# Filter for players with significant playing time (>500 minutes)
player_ratings_filtered = player_ratings.filter(pl.col('estimated_total_minutes') >= 500)

print(f" {len(player_ratings_filtered)} players with >500 minutes")

print("\nTop 10 by attack per-90:")
print(player_ratings_filtered
      .select(['player_name', 'position', 'attack_per90', 'estimated_total_minutes', 'matches_played'])
      .sort('attack_per90', descending=True)
      .head(10))

print("\nTop 10 by defense per-90:")
print(player_ratings_filtered
      .select(['player_name', 'position', 'defense_per90', 'estimated_total_minutes', 'matches_played'])
      .sort('defense_per90', descending=True)
      .head(10))

print("\nTop 10 by passing per-90:")
print(player_ratings_filtered
      .select(['player_name', 'position', 'passing_per90', 'estimated_total_minutes', 'matches_played'])
      .sort('passing_per90', descending=True)
      .head(10))

Normalizing ratings to per-90 minutes...
 396 players with >500 minutes

Top 10 by attack per-90:
shape: (10, 5)
┌────────────────────┬────────────┬──────────────┬─────────────────────────┬────────────────┐
│ player_name        ┆ position   ┆ attack_per90 ┆ estimated_total_minutes ┆ matches_played │
│ ---                ┆ ---        ┆ ---          ┆ ---                     ┆ ---            │
│ str                ┆ str        ┆ f64          ┆ f64                     ┆ u32            │
╞════════════════════╪════════════╪══════════════╪═════════════════════════╪════════════════╡
│ Granit Xhaka       ┆ midfielder ┆ 93.654545    ┆ 2970.0                  ┆ 33             │
│ Joshua Kimmich     ┆ midfielder ┆ 88.810714    ┆ 2520.0                  ┆ 28             │
│ Min-jae Kim        ┆ defender   ┆ 83.6         ┆ 2250.0                  ┆ 25             │
│ Nico Schlotterbeck ┆ defender   ┆ 81.072727    ┆ 2970.0                  ┆ 33             │
│ Angelo Stiller     ┆ midfielder ┆ 78.00

### Scale to 0-100 Rating

Transform per-90 values to a 0-100 scale using MinMaxScaler for intuitive interpretation, where 100 represents the best performer and 0 represents the lowest.

In [17]:
# Cell 14: Scale ratings to 0-100
from sklearn.preprocessing import MinMaxScaler

print("Scaling ratings to 0-100 scale...")

# Extract data for scaling
attack_per90 = player_ratings_filtered['attack_per90'].to_numpy().reshape(-1, 1)
defense_per90 = player_ratings_filtered['defense_per90'].to_numpy().reshape(-1, 1)
passing_per90 = player_ratings_filtered['passing_per90'].to_numpy().reshape(-1, 1)

# Scale to 0-100
scaler_attack = MinMaxScaler(feature_range=(0, 100))
scaler_defense = MinMaxScaler(feature_range=(0, 100))
scaler_passing = MinMaxScaler(feature_range=(0, 100))

attack_rating = scaler_attack.fit_transform(attack_per90).flatten()
defense_rating = scaler_defense.fit_transform(defense_per90).flatten()
passing_rating = scaler_passing.fit_transform(passing_per90).flatten()

# Add scaled ratings
player_ratings_filtered = player_ratings_filtered.with_columns([
    pl.Series('attack_rating', attack_rating),
    pl.Series('defense_rating', defense_rating),
    pl.Series('passing_rating', passing_rating),
])

print(" Ratings scaled to 0-100")

print("\nRating distributions:")
print(f"  Attack:  min={player_ratings_filtered['attack_rating'].min():.1f}, "
      f"max={player_ratings_filtered['attack_rating'].max():.1f}, "
      f"mean={player_ratings_filtered['attack_rating'].mean():.1f}")
print(f"  Defense: min={player_ratings_filtered['defense_rating'].min():.1f}, "
      f"max={player_ratings_filtered['defense_rating'].max():.1f}, "
      f"mean={player_ratings_filtered['defense_rating'].mean():.1f}")
print(f"  Passing: min={player_ratings_filtered['passing_rating'].min():.1f}, "
      f"max={player_ratings_filtered['passing_rating'].max():.1f}, "
      f"mean={player_ratings_filtered['passing_rating'].mean():.1f}")

print("\nTop 5 by each rating:")
print("\nAttack Rating:")
print(player_ratings_filtered
      .select(['player_name', 'position', 'attack_rating', 'matches_played'])
      .sort('attack_rating', descending=True)
      .head(5))

print("\nDefense Rating:")
print(player_ratings_filtered
      .select(['player_name', 'position', 'defense_rating', 'matches_played'])
      .sort('defense_rating', descending=True)
      .head(5))

print("\nPassing Rating:")
print(player_ratings_filtered
      .select(['player_name', 'position', 'passing_rating', 'matches_played'])
      .sort('passing_rating', descending=True)
      .head(5))

Scaling ratings to 0-100 scale...
 Ratings scaled to 0-100

Rating distributions:
  Attack:  min=0.0, max=100.0, mean=28.9
  Defense: min=0.0, max=100.0, mean=32.5
  Passing: min=0.0, max=100.0, mean=25.7

Top 5 by each rating:

Attack Rating:
shape: (5, 4)
┌────────────────────┬────────────┬───────────────┬────────────────┐
│ player_name        ┆ position   ┆ attack_rating ┆ matches_played │
│ ---                ┆ ---        ┆ ---           ┆ ---            │
│ str                ┆ str        ┆ f64           ┆ u32            │
╞════════════════════╪════════════╪═══════════════╪════════════════╡
│ Granit Xhaka       ┆ midfielder ┆ 100.0         ┆ 33             │
│ Joshua Kimmich     ┆ midfielder ┆ 94.561315     ┆ 28             │
│ Min-jae Kim        ┆ defender   ┆ 88.710692     ┆ 25             │
│ Nico Schlotterbeck ┆ defender   ┆ 85.873053     ┆ 33             │
│ Angelo Stiller     ┆ midfielder ┆ 82.433841     ┆ 31             │
└────────────────────┴────────────┴───────────────┴─

### Calculate Position-Weighted Overall Rating

Combine attack, defense, and passing ratings using position-specific weights to create final overall ratings that fairly evaluate players within their tactical roles.

In [18]:
# Cell 15: Calculate position-weighted overall rating
print("Calculating position-specific overall ratings...")

# Define overall rating calculation with position-specific weights
def calculate_overall_rating(row):
    """
    Calculate overall rating by applying position-specific weights.
    
    Forwards: Prioritize attack (60%), passing (30%), defense (10%)
    Midfielders: Balanced across all three (35/35/30)
    Defenders: Prioritize defense (60%), passing (25%), attack (15%)
    """
    position = row['position']
    attack = row['attack_rating']
    defense = row['defense_rating']
    passing = row['passing_rating']
    
    # Get position weights (default to midfielder if position missing)
    weights = POSITION_WEIGHTS.get(position, POSITION_WEIGHTS['midfielder'])
    
    # Calculate weighted average
    overall = (
        attack * weights['attack'] +
        defense * weights['defense'] +
        passing * weights['passing']
    )
    
    return overall

# Apply to all filtered players
overall_ratings = []
for row in player_ratings_filtered.iter_rows(named=True):
    overall_ratings.append(calculate_overall_rating(row))

player_ratings_filtered = player_ratings_filtered.with_columns([
    pl.Series('overall_rating', overall_ratings)
])

print("Overall ratings calculated")

# Display weighting schema
print("\nPosition-specific weighting schema:")
for pos, weights in POSITION_WEIGHTS.items():
    if pos != 'goalkeeper':  # Skip goalkeepers (not in dataset)
        print(f"  {pos:12s}: Attack={weights['attack']:.0%}, "
              f"Passing={weights['passing']:.0%}, "
              f"Defense={weights['defense']:.0%}")

# Show overall top 20
print("TOP 20 PLAYERS - OVERALL RATING")
print(player_ratings_filtered
      .select(['player_name', 'position', 'overall_rating', 
               'attack_rating', 'defense_rating', 'passing_rating', 
               'matches_played'])
      .sort('overall_rating', descending=True)
      .head(20))

# Show top 10 by position

for position in ['forward', 'midfielder', 'defender']:
    print(f"\nTop 10 {position}s:")
    print(player_ratings_filtered
          .filter(pl.col('position') == position)
          .select(['player_name', 'overall_rating', 'attack_rating', 
                   'defense_rating', 'passing_rating', 'matches_played'])
          .sort('overall_rating', descending=True)
          .head(10))

print("Final ratings complete")

Calculating position-specific overall ratings...
Overall ratings calculated

Position-specific weighting schema:
  forward     : Attack=60%, Passing=30%, Defense=10%
  midfielder  : Attack=35%, Passing=35%, Defense=30%
  defender    : Attack=15%, Passing=25%, Defense=60%
TOP 20 PLAYERS - OVERALL RATING
shape: (20, 7)
┌──────────────┬────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ player_name  ┆ position   ┆ overall_rati ┆ attack_rati ┆ defense_rat ┆ passing_rat ┆ matches_pla │
│ ---          ┆ ---        ┆ ng           ┆ ng          ┆ ing         ┆ ing         ┆ yed         │
│ str          ┆ str        ┆ ---          ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆            ┆ f64          ┆ f64         ┆ f64         ┆ f64         ┆ u32         │
╞══════════════╪════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Min-jae Kim  ┆ defender   ┆ 85.106859    ┆ 88.710692   ┆ 82.16547    ┆ 90

### Position-Normalized Overall Ratings

Scale overall ratings to 0-100 within each position group, ensuring the best forward, midfielder, and defender all reach 100 for fair cross-position comparison.

In [19]:
# Cell 15B: Create position-normalized overall ratings
print("Creating position-normalized overall ratings...")
print("="*60)

position_normalized_ratings = []

# Scale overall ratings within each position to 0-100
for position in ['forward', 'midfielder', 'defender']:
    # Get players in this position
    position_players = player_ratings_filtered.filter(pl.col('position') == position)
    
    if len(position_players) == 0:
        continue
    
    # Scale their overall ratings to 0-100 within position
    scaler = MinMaxScaler(feature_range=(0, 100))
    position_overall = position_players['overall_rating'].to_numpy().reshape(-1, 1)
    normalized = scaler.fit_transform(position_overall).flatten()
    
    # Store with player_id
    for i, player_id in enumerate(position_players['player_id']):
        position_normalized_ratings.append({
            'player_id': player_id,
            'position_normalized_rating': normalized[i]
        })

# Join back to main dataframe
position_norm_df = pl.DataFrame(position_normalized_ratings)
player_ratings_filtered = player_ratings_filtered.join(position_norm_df, on='player_id', how='left')

print("Position-normalized ratings added")

# Display examples from each position
print("\n" + "="*60)
print("POSITION-NORMALIZED RATINGS")
print("="*60)

for position in ['forward', 'midfielder', 'defender']:
    print(f"\nTop 5 {position}s (position-normalized):")
    print(player_ratings_filtered
          .filter(pl.col('position') == position)
          .select(['player_name', 'overall_rating', 'position_normalized_rating'])
          .sort('position_normalized_rating', descending=True)
          .head(5))

print("\n" + "="*60)
print("Position normalization complete")
print("="*60)
print("\nNote: Best player in each position now has rating of 100.0")
print("This enables fair comparison across different tactical roles.")

Creating position-normalized overall ratings...
Position-normalized ratings added

POSITION-NORMALIZED RATINGS

Top 5 forwards (position-normalized):
shape: (5, 3)
┌─────────────────┬────────────────┬────────────────────────────┐
│ player_name     ┆ overall_rating ┆ position_normalized_rating │
│ ---             ┆ ---            ┆ ---                        │
│ str             ┆ f64            ┆ f64                        │
╞═════════════════╪════════════════╪════════════════════════════╡
│ Florian Wirtz   ┆ 53.9452        ┆ 100.0                      │
│ Jonas Hofmann   ┆ 47.016897      ┆ 86.928234                  │
│ Xavi Simons     ┆ 45.748122      ┆ 84.534412                  │
│ Leroy Sané      ┆ 43.592084      ┆ 80.466574                  │
│ Andrej Kramaric ┆ 43.049499      ┆ 79.442868                  │
└─────────────────┴────────────────┴────────────────────────────┘

Top 5 midfielders (position-normalized):
shape: (5, 3)
┌───────────────────┬────────────────┬────────────────

In [20]:
# Cell 16: Save final player ratings
print("Saving final player ratings...")

# Save to processed directory
player_ratings_filtered.write_parquet(PROCESSED_DIR / "player_ratings_final.parquet")
player_ratings_filtered.write_csv(PROCESSED_DIR / "player_ratings_final.csv")

print(f" Saved to:")
print(f"   {PROCESSED_DIR / 'player_ratings_final.parquet'}")
print(f"   {PROCESSED_DIR / 'player_ratings_final.csv'}")

print(f"\nFinal dataset: {len(player_ratings_filtered)} players")
print(f"\nColumns: {player_ratings_filtered.columns}")

Saving final player ratings...
 Saved to:
   ../data/processed/player_ratings_final.parquet
   ../data/processed/player_ratings_final.csv

Final dataset: 402 players

Columns: ['player_id', 'team_id', 'total_attack_points', 'total_defense_points', 'total_passing_points', 'total_events', 'matches_played', 'avg_x_position', 'avg_y_position', 'player_name', 'jersey_no', 'total_points', 'attack_ratio', 'defense_ratio', 'position', 'events_per_match', 'estimated_minutes_per_match', 'estimated_total_minutes', 'attack_per90', 'defense_per90', 'passing_per90', 'attack_rating', 'defense_rating', 'passing_rating', 'overall_rating', 'position_normalized_rating']
